# Image Classification (PyTorch)

- Clean corrupted images  
- Preprocess data  
- Split: Train / Val / Test  
- Handle imbalance (Weighted Sampler)  
- EfficientNet-B0 (modified)  
- Train with Early Stopping  
- Evaluate: Accuracy & MCC  

**Output:** best.pth + final results

In [2]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, matthews_corrcoef

DATASET_PATH = r'Data\Industrial-Equipment'
IMG_SIZE = 224
BATCH_SIZE = 16
DEVICE = torch.device("cpu")
MAX_EPOCHS = 8
PATIENCE = 2

print(f"🖥️ Device: {DEVICE}")

def clean_dataset(path):
    for root, _, files in os.walk(path):
        for f in files:
            full = os.path.join(root, f)
            try:
                img = Image.open(full)
                img.verify()
            except:
                try: os.remove(full)
                except: pass

clean_dataset(DATASET_PATH)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

full_ds = datasets.ImageFolder(DATASET_PATH, transform=transform)
class_names = full_ds.classes
num_classes = len(class_names)

train_size = int(0.7 * len(full_ds))
val_size   = int(0.15 * len(full_ds))
test_size  = len(full_ds) - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_ds, [train_size, val_size, test_size])

labels = [full_ds.samples[i][1] for i in train_ds.indices]
class_count = np.bincount(labels)
weights = 1. / class_count
samples_weight = [weights[l] for l in labels]

sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

model = models.efficientnet_b0(weights="IMAGENET1K_V1")

for param in model.features.parameters():
    param.requires_grad = False

in_f = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_f, num_classes)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch():
    model.train()
    preds, y = [], []
    loop = tqdm(train_loader)

    for imgs, labels in loop:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)

        loss.backward()
        optimizer.step()

        _, p = torch.max(out,1)
        preds.extend(p.cpu().numpy())
        y.extend(labels.cpu().numpy())

        loop.set_description(f"loss {loss.item():.4f}")

    return accuracy_score(y, preds)

def evaluate(loader):
    model.eval()
    preds, y = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            out = model(imgs)

            _, p = torch.max(out,1)
            preds.extend(p.cpu().numpy())
            y.extend(labels.numpy())

    return accuracy_score(y, preds)

best = 0
patience_counter = 0

print("\n🚀 Training...")

for epoch in range(MAX_EPOCHS):
    train_acc = train_epoch()
    val_acc   = evaluate(val_loader)

    print(f"Epoch {epoch+1} | Train {train_acc:.4f} | Val {val_acc:.4f}")

    if val_acc > best:
        best = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best.pth")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("⛔ Early stopping")
        break

model.load_state_dict(torch.load("best.pth", weights_only=True))
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        out = model(imgs)

        _, p = torch.max(out,1)
        y_pred.extend(p.cpu().numpy())
        y_true.extend(labels.numpy())

acc = accuracy_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)

print("\n✅ FINAL RESULTS")
print(f"Accuracy: {acc:.4f}")
print(f"MCC: {mcc:.4f}")

🖥️ Device: cpu

🚀 Training...


loss 0.2433: 100%|██████████| 302/302 [04:08<00:00,  1.22it/s]


Epoch 1 | Train 0.9173 | Val 0.9690


loss 0.1037: 100%|██████████| 302/302 [04:00<00:00,  1.26it/s]


Epoch 2 | Train 0.9587 | Val 0.9835


...

Epoch 3 | Train 0.9674 | Val 0.9923


loss 0.0560: 100%|██████████| 302/302 [04:12<00:00,  1.19it/s]


Epoch 4 | Train 0.9704 | Val 0.9923


loss 0.0224: 100%|██████████| 302/302 [04:03<00:00,  1.24it/s]


Epoch 5 | Train 0.9674 | Val 0.9903
⛔ Early stopping

✅ FINAL RESULTS
Accuracy: 0.9884
MCC: 0.9768


# Image Prediction (GUI)

- Select images using file dialog  
- Load & preprocess each image  
- Run model inference  
- Get prediction + confidence  
- Print results for each image  

- if no images → show message  
- else → process all images

In [ ]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image
import torch

root = tk.Tk()
root.withdraw()

file_paths = filedialog.askopenfilenames(
    title="Select Images",
    filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp")]
)

if len(file_paths) == 0:
    print("❌ No images selected")
else:
    for i, file_path in enumerate(file_paths, 1):

        img = Image.open(file_path).convert("RGB")
        img = transform(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(img)
            probs = torch.softmax(output, dim=1)
            conf, pred = torch.max(probs, 1)

        pred_class = class_names[pred.item()]
        confidence = conf.item()

        print(f"\n📸 Image {i}: {file_path}")
        print(f"✅ Prediction: {pred_class}")
        print(f"📊 Confidence: {confidence*100:.2f}%")